Challenge: Real-Time Weather Logger (API + CSV)

Build a Python CLI tool that fetches real-time weather data for a given city and logs it to a CSV file for daily tracking.

Your program should:
1. Ask the user for a city name.
2. Fetch weather data using the OpenWeatherMap API.
3. Store the following in a CSV file (`weather_log.csv`):
   - Date (auto-filled as today's date)
   - City
   - Temperature (in °C)
   - Weather condition (e.g., Clear, Rain)
4. Prevent duplicate entries for the same city on the same day.
5. Allow users to:
   - Add new weather log
   - View all logs
   - Show average, highest, lowest temperatures, and most frequent condition

Bonus:
- Format the output like a table
- Handle API failures and invalid city names gracefully


In [ ]:
import requests

API_KEY = "e7b129d28b2123303f5fe7f23da72706"

url = f"https://api.openweathermap.org/data/2.5/weather?q=Pune&appid={API_KEY}&units=metric"

response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    print(data)
print(response.text)

{'coord': {'lon': 73.8553, 'lat': 18.5196}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 26.83, 'feels_like': 26.74, 'temp_min': 26.83, 'temp_max': 26.83, 'pressure': 1013, 'humidity': 40, 'sea_level': 1013, 'grnd_level': 941}, 'visibility': 10000, 'wind': {'speed': 2.5, 'deg': 332, 'gust': 2.17}, 'clouds': {'all': 61}, 'dt': 1775276255, 'sys': {'country': 'IN', 'sunrise': 1775264187, 'sunset': 1775308711}, 'timezone': 19800, 'id': 1259229, 'name': 'Pune', 'cod': 200}
{"coord":{"lon":73.8553,"lat":18.5196},"weather":[{"id":803,"main":"Clouds","description":"broken clouds","icon":"04d"}],"base":"stations","main":{"temp":26.83,"feels_like":26.74,"temp_min":26.83,"temp_max":26.83,"pressure":1013,"humidity":40,"sea_level":1013,"grnd_level":941},"visibility":10000,"wind":{"speed":2.5,"deg":332,"gust":2.17},"clouds":{"all":61},"dt":1775276255,"sys":{"country":"IN","sunrise":1775264187,"sunset":1775308711},"tim

In [ ]:
import requests
import json
import csv

API_KEY = "e7b129d28b2123303f5fe7f23da72706"

def get_weather(city):
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"
    try:
        response = requests.get(url)    #Sends a GET request to the API
        data = response.json()   #Converts the response into a Python dictionar - API returns JSON
        print(json.dumps(data,indent=3)) #return it as json string

        if response.status_code != 200:    #Checks if request was successful
            print("City not found or API error.")
            return None
        temp = data['main']['temp']
        print('temp:',temp)
        condition = data['weather'][0]['main']
        print('condition:',condition)
        return temp, condition

    except Exception as e:
        print("Error fetching data:", e)
        return None

def log_weather(city, temp, condition):
          date = datetime.now().strftime("%Y-%m-%d")
          exists = os.path.exists("weather_log.csv")

          # Check duplicates
          if exists:
              with open("weather_log.csv", 'r') as f:
                  reader = csv.reader(f)
                  for row in reader:
                      print('\n',row)
                      if row and row[0] == date and row[1].lower() == city.lower():
                          print("Entry already exists for today.")
                          return

          with open("weather_log.csv", 'a') as f: #if not, it creates a new
              writer = csv.writer(f)
              if not exists:
                  writer.writerow(["Date", "City", "Temperature", "Condition"])
              writer.writerow([date, city, temp, condition])
          print("Weather logged successfully...!")

get_weather('Pune')

{
   "coord": {
      "lon": 73.8553,
      "lat": 18.5196
   },
   "weather": [
      {
         "id": 802,
         "main": "Clouds",
         "description": "scattered clouds",
         "icon": "03d"
      }
   ],
   "base": "stations",
   "main": {
      "temp": 32.86,
      "feels_like": 39.86,
      "temp_min": 32.86,
      "temp_max": 32.86,
      "pressure": 1012,
      "humidity": 99,
      "sea_level": 1012,
      "grnd_level": 940
   },
   "visibility": 10000,
   "wind": {
      "speed": 2.73,
      "deg": 309,
      "gust": 3.25
   },
   "clouds": {
      "all": 38
   },
   "dt": 1775281817,
   "sys": {
      "type": 2,
      "id": 2107474,
      "country": "IN",
      "sunrise": 1775264187,
      "sunset": 1775308711
   },
   "timezone": 19800,
   "id": 1259229,
   "name": "Pune",
   "cod": 200
}
temp: 32.86
condition: Clouds


(32.86, 'Clouds')

In [ ]:
from datetime import datetime
import csv
import os
from collections import Counter
from tabulate import tabulate
class WeatherLogger:
    def __init__(self):
      self.menu()

    def menu(self):
      while True:
          print("\n--- Weather Logger ---")
          print("1. Add Weather Log")
          print("2. View Logs")
          print("3. Show Stats")
          print("4. Exit")
          try:
            choice = int(input("Enter choice: "))
          except ValueError:
            print("Invalid choice.")
            continue

          if choice == 1:
              try:
                no=int(input('How many cities to add:'))
              except ValueError:
                print("Invalid choice.")
                continue

              for i in range(no):
                city = input("Enter city: ")
                result = get_weather(city)
                if result:
                    temp, condition = result
                    log_weather(city, temp, condition)
          elif choice == 2:
              self.view_logs()
          elif choice == 3:
              self.show_stats()
          elif choice == 4:
              self.exiting()
              break
          else:
              print("Invalid choice.")


    def view_logs(self):
        if not os.path.exists("weather_log.csv"):
            print("No data available.")
            return

        with open("weather_log.csv", 'r') as f:
            reader = csv.reader(f)
            print("\n-------------------- Weather Logs ---------------------")
            print(tabulate(reader, headers="firstrow", tablefmt="grid"))
        print("Logs viewed successfully...!")


    def show_stats(self):
        if not os.path.exists("weather_log.csv"):
            print("No data available.")
            return

        temps = []
        conditions = []
        with open("weather_log.csv", 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                temps.append(float(row['Temperature']))
                conditions.append(row['Condition'])
        if not temps:
            print("No data to analyze.")
            return
        avg = sum(temps) / len(temps)
        highest = max(temps)
        lowest = min(temps)
        common = Counter(conditions).most_common(1)[0][0]
        print(f"Average Temp: {avg:.2f}°C")
        print(f"Highest Temp: {highest}°C")
        print(f"Lowest Temp: {lowest}°C")
        print(f"Most Common Condition: {common}")

    def exiting(self):
        print("Thank for using waether logger...")
        exit()

w=WeatherLogger()


--- Weather Logger ---
1. Add Weather Log
2. View Logs
3. Show Stats
4. Exit
Enter choice: 2

-------------------- Weather Logs ---------------------
+------------+----------+---------------+-------------+
| Date       | City     |   Temperature | Condition   |
+============+==========+===============+=============+
| 2026-04-04 | Solapur  |         33.18 | Clouds      |
+------------+----------+---------------+-------------+
| 2026-04-04 | Pune     |         28.61 | Clouds      |
+------------+----------+---------------+-------------+
| 2026-04-04 | Baramati |         29.66 | Clouds      |
+------------+----------+---------------+-------------+
Logs viewed successfully...!

--- Weather Logger ---
1. Add Weather Log
2. View Logs
3. Show Stats
4. Exit
Enter choice: 3
Average Temp: 30.48°C
Highest Temp: 33.18°C
Lowest Temp: 28.61°C
Most Common Condition: Clouds

--- Weather Logger ---
1. Add Weather Log
2. View Logs
3. Show Stats
4. Exit
Enter choice: 4
Thank for using waether logger...